# Exploratory Data Analysis - Online Retail Demand Dataset
> **Project:** AI-Based Product Demand Forecasting System  
> **Dataset:** Online Retail (UCI ML Repository)  
> **Objective:** Understand the structure, quality, and patterns in the retail transaction data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')
print('Libraries loaded successfully.')

In [ ]:
# Load the raw Online Retail dataset
df = pd.read_csv('../notebook/data/data.csv', encoding='latin1')

print('Dataset shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

In [ ]:
# ── Basic information ──────────────────────────────────────────
print('Data Types:')
print(df.dtypes)
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
# Detailed schema info
df.info()

In [ ]:
# ── Missing value analysis ──────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print('Columns with missing values:')
print(missing_df)

# Visualise
if not missing_df.empty:
    missing_df['Missing %'].plot(kind='bar', color='salmon', edgecolor='black')
    plt.title('Missing Value Percentage per Column')
    plt.ylabel('Missing %')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Data quality: negatives and cancellations ───────────────────
neg_qty = df[df['Quantity'] < 0]
cancellations = df[df['InvoiceNo'].astype(str).str.startswith('C')]

print(f'Negative quantity rows  : {len(neg_qty):,}')
print(f'Cancellation invoices   : {len(cancellations):,}')
print(f'Rows with missing CustID: {df["CustomerID"].isnull().sum():,}')
print(f'Zero unit-price rows    : {(df["UnitPrice"] == 0).sum():,}')

In [ ]:
# ── Date analysis ───────────────────────────────────────────────
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], infer_datetime_format=True)

print('Date range:')
print(f'  Start : {df["InvoiceDate"].min()}')
print(f'  End   : {df["InvoiceDate"].max()}')
print(f'  Span  : {(df["InvoiceDate"].max() - df["InvoiceDate"].min()).days} days')

In [ ]:
# ── Quantity distribution ───────────────────────────────────────
valid = df[(df['Quantity'] > 0) & (~df['InvoiceNo'].astype(str).str.startswith('C'))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(valid['Quantity'].clip(upper=200), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Quantity Distribution (clipped at 200)')
axes[0].set_xlabel('Quantity')
axes[0].set_ylabel('Frequency')

# Box plot
axes[1].boxplot(valid['Quantity'].clip(upper=200), vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightcoral'))
axes[1].set_title('Quantity Box Plot (clipped at 200)')
axes[1].set_xlabel('Quantity')

plt.tight_layout()
plt.show()

print('\nQuantity statistics (valid transactions):')
print(valid['Quantity'].describe())

In [ ]:
# ── Top 10 countries by total quantity sold ─────────────────────
top_countries = (
    valid.groupby('Country')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_countries.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Top 10 Countries by Total Quantity Sold')
plt.xlabel('Country')
plt.ylabel('Total Quantity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(top_countries)

In [ ]:
# ── Monthly demand trend ────────────────────────────────────────
valid['YearMonth'] = valid['InvoiceDate'].dt.to_period('M')
monthly = valid.groupby('YearMonth')['Quantity'].sum()

monthly.plot(kind='line', marker='o', linewidth=2, color='darkorange')
plt.title('Monthly Total Demand Trend')
plt.xlabel('Month')
plt.ylabel('Total Quantity Sold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Day-of-week analysis ────────────────────────────────────────
valid['DayOfWeek'] = valid['InvoiceDate'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_demand = valid.groupby('DayOfWeek')['Quantity'].sum().reindex(dow_order)

dow_demand.plot(kind='bar', color='mediumslateblue', edgecolor='black')
plt.title('Total Demand by Day of Week')
plt.xlabel('Day')
plt.ylabel('Total Quantity')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 15 products by total quantity ───────────────────────────
top_products = (
    valid.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

top_products.plot(kind='barh', color='cadetblue', edgecolor='black')
plt.title('Top 15 Products by Total Quantity Sold')
plt.xlabel('Total Quantity')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Price vs Quantity correlation ───────────────────────────────
sample = valid[['UnitPrice', 'Quantity']].sample(min(5000, len(valid)), random_state=42)

corr = sample.corr().iloc[0, 1]
print(f'Pearson correlation (UnitPrice vs Quantity): {corr:.4f}')

plt.scatter(sample['UnitPrice'].clip(upper=50), sample['Quantity'].clip(upper=200),
            alpha=0.3, s=10, color='royalblue')
plt.title('Unit Price vs Quantity Sold (clipped)')
plt.xlabel('Unit Price (£)')
plt.ylabel('Quantity')
plt.tight_layout()
plt.show()

In [ ]:
# ── Seasonal decomposition (visual summary) ─────────────────────
from statsmodels.tsa.seasonal import seasonal_decompose

# Use daily aggregation on UK data only for cleaner signal
uk = valid[valid['Country'] == 'United Kingdom'].copy()
uk['Date'] = uk['InvoiceDate'].dt.date
daily_uk = uk.groupby('Date')['Quantity'].sum().asfreq('D', fill_value=0)

result = seasonal_decompose(daily_uk, model='additive', period=7)
result.plot()
plt.suptitle('Seasonal Decomposition – UK Daily Demand', y=1.02)
plt.tight_layout()
plt.show()

## EDA Conclusions

| Finding | Detail |
|---------|--------|
| Dataset span | ~13 months of transactions |
| Dominant market | United Kingdom (~90 % of volume) |
| Cancellations | ~2 % of invoices start with 'C' |
| Missing CustomerID | ~25 % of rows |
| Seasonality | Clear weekly cycle; Q4 peak in November |
| Skewed Quantity | Heavy right-tail; outliers need treatment |

> **Next step:** Clean the data and build the preprocessing pipeline (Notebook 02).